# Practical Data Preprocessing Assignment — Notebook
**Student Name**: Rashmin Dudhatra  
**Course**: Data Preprocessing & Analytics  
**Dataset**: 

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

%matplotlib inline

# Task 1 - Dataset Understanding

In [2]:
# Load dataset directly from GitHub repository
url = 'https://raw.githubusercontent.com/rashmindudhatra001/dpp/main/mall_customer_preprocessing_dataset.csv'
df = pd.read_csv(url)

print('Dataset Dimensions:', df.shape)
df.head()

In [3]:
# Inspect dataset attributes, data types, and summary statistics
print('Attributes List:
', df.columns.tolist())
print('
Data Types:
', df.dtypes)
df.describe(include='all')

### Brief Overview of Dataset Attributes
This dataset contains e-commerce customer transaction histories and demographic profiles:
- **Demographics**: Customer ID, Name, Age, Gender, City, Country.
- **Behavioral Metrics**: Annual Income, Spending Score, Visit Frequency, Avg Basket Value, Total Purchases, Online/Store Purchases.
- **Account Details**: Membership Tier, Join Date, Last Purchase Date, Payment Method, Loyalty Points, Coupon Used, and Churn status.

Numerical attributes describe customer spend and volume, while categorical attributes capture preferences and user demographics.

# Task 2 - Data Quality Assessment

In [4]:
# Check missing values per column and count duplicate records
print('Missing Values per Attribute:')
print(df.isnull().sum()[df.isnull().sum() > 0])

print('
Duplicate Rows:', df.duplicated().sum())
print('Duplicate CustomerIDs:', df['CustomerID'].duplicated().sum())

### 2.1 Summary Table for Data Quality Assessment

In [5]:
# Create a summary table of data quality issues found in the dataset
t2_summary = pd.DataFrame([
    {'Problem': 'Missing Values', 'Column': 'Age, Income, Gender, City, Tier, LastPurchaseDate', 'Count': int(df.isnull().sum().sum()), 'Impact': 'May cause incomplete analysis if not imputed'},
    {'Problem': 'Duplicate Records', 'Column': 'CustomerID / Entire dataset', 'Count': int(df['CustomerID'].duplicated().sum()), 'Impact': 'Over-represents customer metrics'},
    {'Problem': 'Inconsistent Values', 'Column': 'Gender, City', 'Count': 40, 'Impact': 'Same categories treated as distinct due to casing/whitespace'},
    {'Problem': 'Invalid Entries', 'Column': 'Age (-5, 122), Income', 'Count': int(((df['Age'] < 18) | (df['Age'] > 90)).sum()), 'Impact': 'Violates logical domain rules'},
    {'Problem': 'Data Type Issues', 'Column': 'JoinDate, LastPurchaseDate', 'Count': 2, 'Impact': 'String formats prevent date calculations'}
])
t2_summary

# Task 3 - Data Cleaning

In [6]:
# Create a copy of the dataset for cleaning
df_clean = df.copy()

## 3.a Missing Value Imputation

In [7]:
# Check missing values before imputation
print('Missing values before imputation:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Define numerical and categorical feature lists for clean reuse
num_cols = ['Age', 'AnnualIncome_INR', 'SpendingScore_1_100', 'AvgBasketValue_INR', 
            'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'LoyaltyPoints']

# Fill missing numerical values with median
for col in ['Age', 'AnnualIncome_INR']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill missing categorical values with mode
for col in ['Gender', 'City', 'MembershipTier']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('
Missing values after imputation:', df_clean.isnull().sum().sum())

*Reason for Imputation*: Median was used for numerical missing values (Age, Income) to avoid bias from extreme skewed values. Mode was used for categorical features.

## 3.b Duplicate Removal

In [8]:
# Remove duplicate records based on CustomerID
print('Duplicate CustomerIDs before:', df_clean.duplicated(subset=['CustomerID']).sum())
df_clean = df_clean.drop_duplicates(subset=['CustomerID'])
print('Duplicate CustomerIDs after:', df_clean.duplicated(subset=['CustomerID']).sum())
print('Dataset shape after deduplication:', df_clean.shape)

## 3.c Category Standardization

In [9]:
# Inspect unique values before standardization
print('Gender before:', df_clean['Gender'].unique())
print('City before:', df_clean['City'].unique()[:5])

# Strip whitespace and standardize capitalization
df_clean['Gender'] = df_clean['Gender'].astype(str).str.strip().str.capitalize()
df_clean['Gender'] = df_clean['Gender'].replace({'M': 'Male', 'Female': 'Female', 'Other': 'Other'})

df_clean['City'] = df_clean['City'].astype(str).str.strip().str.title()

print('
Gender after:', df_clean['Gender'].unique())
print('City after:', df_clean['City'].unique()[:5])

## 3.d Data Type Conversion

In [10]:
# Convert Age to integer and date strings to datetime objects
df_clean['Age'] = df_clean['Age'].astype(int)
df_clean['JoinDate'] = pd.to_datetime(df_clean['JoinDate'], errors='coerce')
df_clean['LastPurchaseDate'] = pd.to_datetime(df_clean['LastPurchaseDate'], errors='coerce')

print('Converted Data Types:')
print(df_clean.dtypes[['Age', 'JoinDate', 'LastPurchaseDate']])

## 3.e Noise Handling

In [11]:
# Replace unrealistic age values (< 18 or > 90) with median age
invalid_age_mask = (df_clean['Age'] < 18) | (df_clean['Age'] > 90)
print('Invalid Age count before:', invalid_age_mask.sum())

median_age = int(df_clean['Age'].median())
df_clean.loc[invalid_age_mask, 'Age'] = median_age

print('Invalid Age count after:', ((df_clean['Age'] < 18) | (df_clean['Age'] > 90)).sum())

## 3.f Outlier Treatment

In [12]:
# Treat extreme outliers in income and basket value using IQR capping
for col in ['AnnualIncome_INR', 'AvgBasketValue_INR']:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap extreme values to IQR limits
    df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)

print('Outlier capping completed using IQR boundaries.')

## 3.g Final Validation

In [13]:
# Verify clean dataset dimensions and missing values count
print('Final Clean Dataset Shape:', df_clean.shape)
print('Total Null Values Remaining:', df_clean.isnull().sum().sum())
print('Total Duplicate Rows Remaining:', df_clean.duplicated(subset=['CustomerID']).sum())

# Save clean dataset locally
df_clean.to_csv('cleaned_mall_customer_dataset.csv', index=False)
print('Clean dataset saved to cleaned_mall_customer_dataset.csv.')

# Task 4 - Data Transformation

## 4.a Label Encoding

In [14]:
# Apply Label Encoding to binary categorical feature CouponUsed
le = LabelEncoder()
df_clean['CouponUsed_Encoded'] = le.fit_transform(df_clean['CouponUsed'].astype(str))
print('CouponUsed Label Encoding Mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

## 4.b One-Hot Encoding

In [15]:
# Apply One-Hot Encoding to categorical attributes with drop_first=True to avoid dummy variable trap
ohe_cols = ['Gender', 'MembershipTier', 'VisitFrequency', 'PreferredCategory', 'City', 'PaymentMethod']
df_ohe = pd.get_dummies(df_clean, columns=[c for c in ohe_cols if c in df_clean.columns], drop_first=True)

print('Shape after One-Hot Encoding:', df_ohe.shape)
df_ohe.head(3)

## 4.c Min-Max Normalization

In [16]:
# Apply Min-Max Normalization to scale numerical columns to [0, 1]
scaler = MinMaxScaler()
df_norm = df_clean.copy()

df_norm[num_cols] = scaler.fit_transform(df_norm[num_cols])
print('Min-Max Normalization applied successfully.')
df_norm[num_cols].head(3)

# Task 5 - Data Reduction

## 5.a Principal Component Analysis (PCA)

In [17]:
# Apply PCA to reduce 8 normalized numerical features down to 5 principal components
pca = PCA(n_components=5)
pca_result = pca.fit_transform(df_norm[num_cols])

df_pca = pd.DataFrame(pca_result, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5'])

print('Explained Variance Ratio per Component:')
print(pca.explained_variance_ratio_)
print('
Total Cumulative Variance Retained:', f"{pca.explained_variance_ratio_.sum() * 100:.2f}%")
df_pca.head()

## 5.b Random Sampling

In [18]:
# Take an 80% random sample of the dataset with a fixed seed
df_sample = df_norm.sample(frac=0.8, random_state=42)
print('Original dataset shape:', df_norm.shape)
print('Sampled dataset shape (80%):', df_sample.shape)

# Task 6 - Proximity Measures

## 6.a Euclidean Distance

In [19]:
# Compute Euclidean distance matrix for first 20 customers
df_20 = df_norm[num_cols].head(20)
euc_dist = pairwise_distances(df_20, metric='euclidean')

plt.figure(figsize=(8, 6))
sns.heatmap(euc_dist, annot=False, cmap='Blues')
plt.title('Euclidean Distance Heatmap (First 20 Customers)')
plt.xlabel('Customer Index')
plt.ylabel('Customer Index')
plt.show()

## 6.b Manhattan Distance

In [20]:
# Compute Manhattan distance matrix for first 20 customers
man_dist = pairwise_distances(df_20, metric='manhattan')

plt.figure(figsize=(8, 6))
sns.heatmap(man_dist, annot=False, cmap='Oranges')
plt.title('Manhattan Distance Heatmap (First 20 Customers)')
plt.xlabel('Customer Index')
plt.ylabel('Customer Index')
plt.show()

Both Euclidean and Manhattan distance heatmaps reveal customer clusters with similar purchasing metrics. Darker cells represent customer pairs with smaller distance values.

# Task 7 - Reflection Report

### 1. Major Preprocessing Challenges
The main challenge was handling multiple data issues together: missing values, text typos and mixed casing in categories (e.g.  vs ), invalid numbers like negative ages, and high income outliers. Using median imputation instead of mean was helpful because income and spending values were skewed.

### 2. Preprocessing Step with Greatest Impact
Data cleaning (Task 3) had the biggest overall impact. Cleaning typos, handling missing values, and capping outliers made sure that later steps like one-hot encoding and Min-Max scaling worked on valid, consistent data.

### 3. Limitations of the Pipeline
- Median imputation fills gaps with a typical value, which might smooth over small customer sub-groups.
- Outlier IQR capping keeps values bounded for scaling, but extreme high spenders get clipped.
- PCA reduces feature count, but principal components are harder to interpret than original column names.

### 4. Recommended Machine Learning Algorithms
- **K-Means Clustering**: Fits well for customer segmentation using scaled features or principal components.
- **Decision Tree / Random Forest Classifier**: Good choices for predicting customer churn () since trees handle tabular data and non-linear patterns effectively.

### 5. Future Preprocessing Improvements
To make this pipeline better for future data, I would write reusable helper functions for the cleaning steps and save the fitted  model object so new incoming test data can be scaled using the exact same min and max parameters.